<a href="https://colab.research.google.com/github/VR952004/Real-time-Anomaly-detection-in-F1/blob/main/Anomaly_Detection_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q fastf1 scikit-learn pandas joblib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.5/135.5 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.2/70.2 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 59.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.9 MB/s eta 0:00:00


In [2]:
import fastf1
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
import joblib
from google.colab import files

fastf1.Cache.enable_cache('.')
print("Environment ready!")

Environment ready!


In [3]:
print("Downloading session data... this might take a minute on the first run.")

session = fastf1.get_session(2026, 'Shanghai', 'Sprint')
session.load()

print("Extracting Max Verstappen's telemetry...")
laps = session.laps.pick_driver('VER')
telemetry = laps.get_telemetry()

print("Extracting and merging Track Status...")
track_status = session.track_status

track_status = track_status.rename(columns={'Time': 'SessionTime'})

telemetry = pd.merge_asof(
    telemetry.sort_values('SessionTime'),
    track_status.sort_values('SessionTime'),
    on='SessionTime',
    direction='backward'
)

print(f"Success! Extracted {len(laps)} clean race-pace laps.")
print(f"Total telemetry rows extracted: {len(telemetry)}")

core           INFO 	Loading data for Chinese Grand Prix - Sprint [v3.8.1]
INFO:fastf1.fastf1.core:Loading data for Chinese Grand Prix - Sprint [v3.8.1]
req            INFO 	No cached data found for session_info. Loading data...
INFO:fastf1.fastf1.req:No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
INFO:fastf1.api:Fetching session info data...


req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req:Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
INFO:fastf1.fastf1.req:No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
INFO:fastf1.api:Fetching driver list...
req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req:Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
INFO:fastf1.fastf1.req:No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
INFO:fastf1.api:Fetching session status data...
req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req:Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
INFO:fastf1.fastf1.req:No cached data found for lap_count. Loading data...
_api           INFO 	F

Extracting Max Verstappen's telemetry...
Extracting and merging Track Status...
Success! Extracted 19 clean race-pace laps.
Total telemetry rows extracted: 15340


In [4]:
if 'Status' not in telemetry.columns:
    telemetry = pd.merge_asof(
        telemetry.sort_values('SessionTime'),
        track_status[['SessionTime', 'Status']].sort_values('SessionTime'),
        on='SessionTime',
        direction='backward'
    )

features = ['Speed', 'RPM', 'nGear', 'Throttle', 'Brake', 'Status']
df = telemetry[features].copy()

# Convert Brake and Status to integers, handling any NaNs from the merge
df['Brake'] = df['Brake'].fillna(0).astype(int)
df['Status'] = df['Status'].fillna(-1).astype(int)

df = df.dropna()

print("Features prepared for model training:")
df.head()

Features prepared for model training:


,Speed,RPM,nGear,Throttle,Brake,Status
0,0.0,11110.624997,1,19.000000,1,1
1,0.0,11111.000000,1,19.000000,1,1
2,0.0,11232.409891,1,19.000000,1,1
3,0.0,11282.000000,1,19.000000,1,1
4,0.0,11083.666916,1,19.283333,1,1


In [5]:
df['Status'].unique()

array([1, 2, 4])

In [6]:
model = IsolationForest(
    n_estimators=1000,
    contamination=0.01,
    random_state=42,
    n_jobs=-1
)

print("Training the model on VER's telemetry")
model.fit(df)

print("Training complete!")

Training the model on VER's telemetry
Training complete!


In [7]:
model_filename = 'f1_anomaly_model.pkl'

joblib.dump(model, model_filename)
print(f"Model saved as {model_filename}")

files.download(model_filename)

Model saved as f1_anomaly_model.pkl


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>